# Introdução à Inteligência Artificial e Fundamentos de Python

## Florestas de decisão  aleatórias(*Random forest*)

Floresta aleatória é um método de aprendizagem de máquina no qual são criadas muitas árvores de decisão para fazer melhores previsões em tarefas de classificação e regressão. Árvores de decisão Árvores de decisão são propensas ao *overfit*. Outra questão (relacionada ao *overfitting*) é que, mesmo alterando um pouco os dados de treinamento, você pode acabar com uma árvore completamente diferente. Por outro lado, uma floresta de decisão contém muitas árvores básicas e prevêem os resultados por meio das médias das previsões individuais de cada árvore. Também, em uma floresta de decisão, o conjunto de dados para treinar cada árvore é amostrado aleatoriamente pelo método de *bootstrapping*.

Considere o exemplo visto anteriormente para previsão da decisão de se jogar ou não tênis com base nas condições climáticas:

| Dia | Céu        | Temperatura | Umidade | Vento | Jogar? |
| --- | ---------- | ----------: | ------: | ----- | ------ |
| 1   | Ensolarado |          30 |      70 | Fraco | Não    |
| 2   | Ensolarado |          28 |      80 | Forte | Não    |
| 3   | Nublado    |          25 |      65 | Fraco | Sim    |
| 4   | Chuvoso    |          20 |      90 | Fraco | Não    |
| 5   | Chuvoso    |          22 |      75 | Fraco | Sim    |
| 6   | Nublado    |          27 |      70 | Forte | Sim    |
| 7   | Ensolarado |          32 |      60 | Fraco | Sim    |
| 8   | Chuvoso    |          23 |      85 | Forte | Não    |
| 9   | Nublado    |          24 |      68 | Fraco | Sim    |
| 10  | Ensolarado |          29 |      72 | Forte | Não    |
 

              Céu?
             /    \
       Nublado   Ensolarado
          |        /     \
         SIM     Umidade  ?               
                 /    \
              <=70    >70
               |       |
              SIM      NÃO

Esta árvore é sujeita a aprender algo muito específico com base nos dados de treino recebidos, ou seja, de sofrer *overfitting*. Uma ideia alternativa é dividir a tarefa de aprendizado, assim como os dados de treino, em várias árvores, e fazer uma contagem de votos. 

<div>
<img src="Files\ChatGPT RandomForest.png" width="500" height="800">   
</div>
Fonte: Gerada com IA (ChatGPT)

Assim, o funcionamento da algoritmo da floresta aleatória pode ser sumarizado como:

* **Criação de várias árvores de decisão:** O algoritmo gera diversas árvores de decisão, utilizando uma parte aleatória dos dados para cada uma. Assim, cada árvore é ligeiramente diferente das outras.
* **Seleção aleatória de características:** Ao construir cada árvore, o algoritmo não analisa todas as características (colunas) de uma só vez; ele seleciona algumass aleatoriamente para decidir como dividir os dados. Isso ajuda a garantir que as árvores sejam diferentes entre si.
* **Cada árvore faz uma previsão:** Cada árvore fornece sua própria resposta ou previsão com base no que aprendeu a partir da sua parcela de dados.
* **Combinação das previsões:** No caso de classificação, a resposta final é a categoria que recebe a maioria dos votos das árvores (votação majoritária).
* **Por que funciona bem:** O uso de dados e características aleatórios para cada árvore ajuda a evitar o *overfitting* (sobreajuste) e torna a previsão geral mais precisa e confiável.


### Implementação de uma Floresta Aleatória para Tarefas de Classificação Usando Scikit Learn

In [1]:
# Importação das bibliotecas
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

1) Leitura dos dados e Criação de um *DataFrame*

In [2]:
titanic_data = pd.read_csv("Files/titanic.csv")
# Eliminação das linhas nas quais 'Survived" esá ausente ou vazio. 
titanic_data = titanic_data.dropna(subset=['Survived'])

2) Escolha das características
   
   X irá conter as características usadas para fazer a previsão, que são:
* Pclass → classe do passageiro
* Sex    → sexo
* Age    → idade
* SibSp  → irmãos/cônjuges a bordo
* Parch  → pais/filhos a bordo
* Fare   → tarifa
  y é a variável alvo ou rótulo (*target*). Neste caso:
* Survived = 1 → sobreviveu
* Survived = 0 → não sobreviveu


In [3]:

X = titanic_data[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']]
y = titanic_data['Survived']

3) Pré-processamento:

In [4]:
# Tranformação do gênero em algarismos, pois o modelo precisa trabalhar com valores numéricos
X['Sex'] = X['Sex'].map({'female': 0, 'male': 1})

# Algumas idades estão ausentes (NaN), e fillna.median() preenche esses campos com a mediana das idades. 
X['Age'] = X['Age'].fillna(X['Age'].median())

# Divisão entre dados de treinamento e teste (80% para treinamento e 20% para teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

4) criação da Floresta Aleatória

In [5]:
# 100 árvores de decisão
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
# treinamento
rf_classifier.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

5) Previsões e medição da acurácia

In [6]:
y_pred = rf_classifier.predict(X_test)

# Comparação dos valores reais de y com valores previstos
accuracy = accuracy_score(y_test, y_pred)

classification_rep = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:\n", classification_rep)

#Escolha de um passageiro (escolhe a linha inteira preservando a esttrutura do dataFrame)
sample = X_test.iloc[123:124]
prediction = rf_classifier.predict(sample)

#Conversão da estrutura em Sample em um dicionário
sample_dict = sample.iloc[0].to_dict()
print(f"\nSample Passenger: {sample_dict}")
print(f"Predicted Survival: {'Survived' if prediction[0] == 1 else 'Did Not Survive'}")

Accuracy: 0.80

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.85      0.83       105
           1       0.77      0.73      0.75        74

    accuracy                           0.80       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179


Sample Passenger: {'Pclass': 2.0, 'Sex': 1.0, 'Age': 0.83, 'SibSp': 0.0, 'Parch': 2.0, 'Fare': 29.0}
Predicted Survival: Survived


### Implementação de uma Floresta Aleatória para Tarefas de Regressão Usando Scikit Learn

* Carrega o conjunto de dados de habitação da Califórnia e cria um DataFrame com as variáveis características (features) e o alvo (target).
* Divide os dados em conjuntos de treinamento e teste (80% para treino, 20% para teste).
* Inicializa e treina um regressor Floresta Aleatória utilizando os dados de treinamento.
* Na regressão, em vez de votação por maioria, as previsões das árvores são combinadas, tipicamente pela média.
* Realiza previsões dos valores das casas nos dados de teste e avalia o desempenho utilizando o MSE e o coeficiente $R^2$.
* Exibe uma previsão de exemplo e compara  com o valor real.

In [7]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

california_housing = fetch_california_housing()
california_data = pd.DataFrame(california_housing.data, columns=california_housing.feature_names)
california_data['MEDV'] = california_housing.target

X = california_data.drop('MEDV', axis=1)
y = california_data['MEDV']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

rf_regressor.fit(X_train, y_train)

y_pred = rf_regressor.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Uma linha de X_test, retira a estrutura do Pandas mantendo apenas valores  e transforma em uma array 2D :1 linha × 8 características
single_data = X_test.iloc[0].values.reshape(1, -1)
predicted_value = rf_regressor.predict(single_data)
print(f"Predicted Value: {predicted_value[0]:.2f}")
print(f"Actual Value: {y_test.iloc[0]:.2f}")

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")

Predicted Value: 0.51
Actual Value: 0.48
Mean Squared Error: 0.26
R-squared Score: 0.81



### Resumo dos Exemplos

| Titanic                  | California Housing             |
| ------------------------ | ------------------------------ |
| `RandomForestClassifier` | `RandomForestRegressor`        |
| classificação            | regressão                      |
| saída: 0 ou 1            | saída: número real             |
| votação das árvores      | combinação/média das previsões |
| `accuracy`               | `MSE`                          |
| `classification_report`  | `R²`                           |
